# 🎬 TMDB Movie Recommendation System: Data Preprocessing


#Imports & Loading

In [12]:
import pandas as pd
import ast
from google.colab import files

# Loading the file
movies = pd.read_csv('/content/tmdb_5000_movies.csv')

# Select columns
# Rename 'id' to 'movie_id'
movies = movies[['id', 'title', 'overview', 'genres', 'keywords']]
movies.rename(columns={'id': 'movie_id'}, inplace=True)

# Drop empty rows
movies.dropna(inplace=True)

print("✅ Data Loaded.")

✅ Data Loaded.


#The Helper Function

In [13]:
def extract_names(text):
    """
    Converts '[{"name": "Science Fiction"}]' -> ['Science Fiction']
    """
    try:
        return [i['name'] for i in ast.literal_eval(text)]
    except ValueError:
        return []

#The Cleaning Stage

In [14]:
# 1. Clean Genres and Keywords into Lists
movies['genres'] = movies['genres'].apply(extract_names)
movies['keywords'] = movies['keywords'].apply(extract_names)

# 2. Prepare the pieces for the 'tags'
# We make NEW temporary variables just for the tags, so we don't mess up the original display.

# A. Overview words (Split)
overview_tags = movies['overview'].apply(lambda x: x.split())

# B. Genre words (Remove spaces for tags: "Science Fiction" -> "ScienceFiction")
genre_tags = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])

# C. Keyword words (Remove spaces)
keyword_tags = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])

# 3. Create the 'tags' column by combining the pieces
movies['tags'] = overview_tags + genre_tags + keyword_tags

# 4. Convert 'tags' to a single lowercase string
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x).lower())

print("✅ Tags created. Original 'overview' and 'genres' are safe!")

✅ Tags created. Original 'overview' and 'genres' are safe!


#Download Dataset

In [15]:
# Select the final columns
# We keep the readable 'overview' and the list-format 'genres' for display
final_df = movies[['movie_id', 'title', 'tags', 'overview', 'genres']]

# Preview to prove it's correct
print("Overview (Readable):", final_df['overview'].iloc[0])
print("Tags ( for AI ):    ", final_df['tags'].iloc[0])

# Save and Download
#final_df.to_csv('final_movies_dataset.csv', index=False)
#files.download('final_movies_dataset.csv')

Overview (Readable): In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.
Tags ( for AI ):     in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d


#Loading the Clean Data

In [16]:
import pandas as pd
df = pd.read_csv('final_movies_dataset.csv')

print("✅ Data Loaded. Ready for the AI.")
df.head(2)

✅ Data Loaded. Ready for the AI.


,movie_id,title,tags,overview,genres
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di...","In the 22nd century, a paraplegic Marine is di...","['Action', 'Adventure', 'Fantasy', 'Science Fi..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha...","Captain Barbossa, long believed to be dead, ha...","['Adventure', 'Fantasy', 'Action']"


#Vectorization (Turning Text into Numbers)

In [17]:
from sklearn.feature_extraction.text import CountVectorizer

# 1. Initialize the Vectorizer
# max_features=5000: We only take the top 5,000 most common words to keep it fast.
# stop_words='english': We remove boring words like "the", "a", "in".
cv = CountVectorizer(max_features=5000, stop_words='english')

# 2. Converting the 'tags' column into a matrix of numbers
# This creates a Grid: Rows = Movies, Columns = The 5000 words
vectors = cv.fit_transform(df['tags']).toarray()

print(f"✅ Vectors created. We have {vectors.shape[0]} movies and {vectors.shape[1]} words describing them.")

✅ Vectors created. We have 4800 movies and 5000 words describing them.


#Calculating Cosine Similarity

In [18]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculating the similarity score of every movie against every other movie
# This will create a huge matrix (4806 x 4806)
similarity = cosine_similarity(vectors)
print(similarity)
print("✅ Similarity Matrix Calculated.")
print(f"Shape: {similarity.shape}")
# Example: similarity[0] is the list of distances between Avatar and every other movie.

[[1.         0.09583148 0.06277648 ... 0.02465568 0.0270666  0.        ]
 [0.09583148 1.         0.07018624 ... 0.02756589 0.         0.        ]
 [0.06277648 0.07018624 1.         ... 0.02708645 0.         0.        ]
 ...
 [0.02465568 0.02756589 0.02708645 ... 1.         0.07007128 0.04732485]
 [0.0270666  0.         0.         ... 0.07007128 1.         0.05195243]
 [0.         0.         0.         ... 0.04732485 0.05195243 1.        ]]
✅ Similarity Matrix Calculated.
Shape: (4800, 4800)


#Building the Recommendation Function

In [19]:
def recommend(movie_title):
    try:
        # 1. Finding the index (row number) of the movie
        # We look for the title in our dataframe
        movie_index = df[df['title'] == movie_title].index[0]

        # 2. Get the similarity scores for this specific movie
        # This returns a list of numbers (0.1, 0.9, 0.0, etc.)
        distances = similarity[movie_index]

        # 3. Sort the movies based on similarity score (Highest match first)
        # enumerate(distances) keeps the original movie ID attached to the score
        # reverse=True means we want highest scores first
        # [1:6] means we skip index 0 (because that's the movie itself) and take the next 5
        movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x:x[1])[1:11]

        # 4. Print the top 5 recommendations
        print(f"🤖 If you liked '{movie_title}', you should watch:")
        print("-" * 40)
        for i in movies_list:
            # i[0] is the movie_id, i[1] is the score
            print(df.iloc[i[0]].title)

    except IndexError:
        print(f"❌ Error: Movie '{movie_title}' not found in the dataset.")

#Test the System!

In [20]:
# Test 1: Superhero Movie
recommend('Batman Begins')

print("\n")

# Test 2:
recommend('Spider-Man 2')

print("\n")

# Test 3: Sci-Fi
recommend('Avatar')



🤖 If you liked 'Batman Begins', you should watch:
----------------------------------------
The Dark Knight
Batman
Batman & Robin
The Dark Knight Rises
Batman v Superman: Dawn of Justice
Amidst the Devil's Wings
Defendor
Dead Man Down
Batman Forever
Batman Returns


🤖 If you liked 'Spider-Man 2', you should watch:
----------------------------------------
Spider-Man 3
Spider-Man
The Amazing Spider-Man 2
The Amazing Spider-Man
Hellboy II: The Golden Army
X-Men
Deadpool
Superman
Batman
Captain America: Civil War


🤖 If you liked 'Avatar', you should watch:
----------------------------------------
Titan A.E.
Small Soldiers
Independence Day
Ender's Game
Aliens vs Predator: Requiem
Battle: Los Angeles
Predators
Lifeforce
Jupiter Ascending
Falcon Rising


In [21]:
import pickle
from google.colab import files

# 1. Save the DataFrame (Movie Data)
# We save it as a dictionary (to_dict) because it's lighter and safer to move around
pickle.dump(df.to_dict(), open('movie_dict.pkl', 'wb'))

# 2. Save the Similarity Matrix (The Math)
pickle.dump(similarity, open('similarity.pkl', 'wb'))

print("✅ Brain saved! Downloading files now...")

# 3. Trigger Download
files.download('movie_dict.pkl')
files.download('similarity.pkl')

✅ Brain saved! Downloading files now...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>